In [1]:
# 1 — KAGGLE ENVIRONMENT + GLOBAL PATH REGISTRY (FINAL)

import os
import gc
import numpy as np
import pandas as pd

# --- FIND COMPETITION ROOT ---
BASE_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = None
for p in BASE_CANDIDATES:
    if os.path.exists(p):
        COMP_ROOT = p
        break

if COMP_ROOT is None:
    raise FileNotFoundError(
        "❌ Competition dataset not found.\n"
        "👉 Click 'Add Input' and attach: Stanford RNA 3D Folding 2"
    )

# --- INPUT FILES ---
TRAIN_LABELS_PATH = os.path.join(COMP_ROOT, "train_labels.csv")
VALIDATION_LABELS_PATH = os.path.join(COMP_ROOT, "validation_labels.csv")
TRAIN_SEQS_PATH = os.path.join(COMP_ROOT, "train_sequences.csv")
VALIDATION_SEQS_PATH = os.path.join(COMP_ROOT, "validation_sequences.csv")
TEST_SEQS_PATH = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_PATH = os.path.join(COMP_ROOT, "sample_submission.csv")

REQUIRED_FILES = [
    TRAIN_LABELS_PATH,
    VALIDATION_LABELS_PATH,
    TRAIN_SEQS_PATH,
    VALIDATION_SEQS_PATH,
    TEST_SEQS_PATH,
    SAMPLE_SUB_PATH,
]

missing_files = [p for p in REQUIRED_FILES if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError("❌ Missing required files:\n" + "\n".join(missing_files))

# --- WORKING OUTPUT PATHS ---
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

FEATURE_OUT = os.path.join(WORK_DIR, "FEATURE_TABLE__GEOMETRY_LABELS_V1.csv")
BENCH_OUT   = os.path.join(WORK_DIR, "HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv")

# --- PRINT STATE ---
print("✅ Competition root:", COMP_ROOT)
print("✅ Working dir:", WORK_DIR)
print("✅ Feature output:", FEATURE_OUT)
print("✅ Benchmark output:", BENCH_OUT)

print("\n📁 Files in competition root:")
for f in os.listdir(COMP_ROOT):
    print(" -", f)

✅ Competition root: /kaggle/input/competitions/stanford-rna-3d-folding-2
✅ Working dir: /kaggle/working
✅ Feature output: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv
✅ Benchmark output: /kaggle/working/HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv

📁 Files in competition root:
 - MSA
 - sample_submission.csv
 - validation_sequences.csv
 - test_sequences.csv
 - validation_labels.csv
 - extra
 - train_labels.csv
 - train_sequences.csv
 - PDB_RNA


In [2]:
# 2 — LOAD COMPETITION LABELS (MEMORY-SAFE, OPTIMIZED)

import pandas as pd
import numpy as np
import gc

USE_COLS = ["ID", "resname", "resid", "x_1", "y_1", "z_1", "chain", "copy"]

DTYPES = {
    "ID": "string",
    "resname": "category",
    "resid": "int32",
    "x_1": "float32",
    "y_1": "float32",
    "z_1": "float32",
    "chain": "category",
    "copy": "category",
}

print("📥 Loading train labels...")
df_labels = pd.read_csv(
    TRAIN_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

print("📥 Appending validation labels...")
df_val = pd.read_csv(
    VALIDATION_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

# Append WITHOUT creating large intermediate copies
df_labels = pd.concat([df_labels, df_val], ignore_index=True)
del df_val
gc.collect()

# --- DERIVE FIELDS ---
df_labels["target_id"] = df_labels["ID"].str.split("_").str[0]

df_labels["residue_index"] = df_labels["resid"].astype(np.int32)

df_labels["x"] = df_labels["x_1"]
df_labels["y"] = df_labels["y_1"]
df_labels["z"] = df_labels["z_1"]

# Drop original heavy columns EARLY
df_labels = df_labels.drop(columns=["ID", "resid", "x_1", "y_1", "z_1"])

gc.collect()

# --- SORT (critical for geometry) ---
df_labels = df_labels.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

gc.collect()

# --- FINAL REPORT ---
print("✅ Label rows loaded:", len(df_labels))
print("✅ Targets:", df_labels["target_id"].nunique())
print("✅ Chains/copies:", df_labels.groupby(["target_id", "chain", "copy"]).ngroups)

print("\n📊 Memory usage (MB):", round(df_labels.memory_usage(deep=True).sum() / 1e6, 2))

df_labels.head(10)

📥 Loading train labels...
📥 Appending validation labels...
✅ Label rows loaded: 7804733
✅ Targets: 5744
✅ Chains/copies: 17952

📊 Memory usage (MB): 1330.14


,resname,chain,copy,target_id,residue_index,x,y,z
0,C,A,1,157D,1,4.843000,-5.640,13.265
1,G,A,1,157D,2,3.385000,-7.613,8.267
2,C,A,1,157D,3,2.158000,-6.751,2.949
3,G,A,1,157D,4,2.669000,-4.843,-1.773
4,A,A,1,157D,5,3.509000,0.239,-4.045
5,A,A,1,157D,6,6.073000,4.823,-5.035
6,U,A,1,157D,7,10.129000,7.706,-4.251
7,U,A,1,157D,8,15.514000,8.745,-2.055
8,A,A,1,157D,9,20.429001,7.281,-1.699
9,G,A,1,157D,10,23.509001,2.728,-1.918


In [3]:
# 3 — ENFORCE SEQUENTIAL GEOMETRY + PHYSICAL STEP FILTER

group_cols = ["target_id", "chain", "copy"]

df_geo = df_labels.copy()

df_geo["resid_diff"] = df_geo.groupby(group_cols)["residue_index"].diff()
df_geo["is_seq"] = df_geo["resid_diff"].eq(1)

df_geo = df_geo[df_geo["resid_diff"].isna() | df_geo["is_seq"]].copy()

df_geo["dx"] = df_geo.groupby(group_cols)["x"].diff()
df_geo["dy"] = df_geo.groupby(group_cols)["y"].diff()
df_geo["dz"] = df_geo.groupby(group_cols)["z"].diff()
df_geo["step"] = np.sqrt(df_geo["dx"]**2 + df_geo["dy"]**2 + df_geo["dz"]**2)

df_geo = df_geo[
    df_geo["step"].isna() | ((df_geo["step"] >= 4.5) & (df_geo["step"] <= 8.0))
].copy()

steps = df_geo["step"].dropna().values
print("✅ Sequential geometry rows:", len(df_geo))
print("✅ Step mean:", float(steps.mean()))
print("✅ Step median:", float(np.median(steps)))
print("✅ Step p90:", float(np.percentile(steps, 90)))
print("✅ Step min:", float(steps.min()))
print("✅ Step max:", float(steps.max()))

display(df_geo.head(10))


✅ Sequential geometry rows: 6996848
✅ Step mean: 5.791539669036865
✅ Step median: 5.574956893920898
✅ Step p90: 7.013174057006836
✅ Step min: 4.500007629394531
✅ Step max: 7.999997615814209


,resname,chain,copy,target_id,residue_index,x,y,z,resid_diff,is_seq,dx,dy,dz,step
0,C,A,1,157D,1,4.843000,-5.640,13.265,NaN,False,NaN,NaN,NaN,NaN
1,G,A,1,157D,2,3.385000,-7.613,8.267,1.0,True,-1.458000,-1.973,-4.998,5.567629
2,C,A,1,157D,3,2.158000,-6.751,2.949,1.0,True,-1.227000,0.862,-5.318,5.525369
3,G,A,1,157D,4,2.669000,-4.843,-1.773,1.0,True,0.511000,1.908,-4.722,5.118483
4,A,A,1,157D,5,3.509000,0.239,-4.045,1.0,True,0.840000,5.082,-2.272,5.629770
5,A,A,1,157D,6,6.073000,4.823,-5.035,1.0,True,2.564000,4.584,-0.990,5.344834
6,U,A,1,157D,7,10.129000,7.706,-4.251,1.0,True,4.056000,2.883,0.784,5.037606
7,U,A,1,157D,8,15.514000,8.745,-2.055,1.0,True,5.385000,1.039,2.196,5.907636
8,A,A,1,157D,9,20.429001,7.281,-1.699,1.0,True,4.915001,-1.464,0.356,5.140746
9,G,A,1,157D,10,23.509001,2.728,-1.918,1.0,True,3.080000,-4.553,-0.219,5.501288


In [4]:
# 4 — BUILD GEOMETRY FEATURES (NORMALIZED DIRECTION + CURVATURE + TURN ANGLE)

df_feat = df_geo.copy()

df_feat["step_safe"] = df_feat["step"].replace(0, np.nan)
df_feat["dx_norm"] = df_feat["dx"] / df_feat["step_safe"]
df_feat["dy_norm"] = df_feat["dy"] / df_feat["step_safe"]
df_feat["dz_norm"] = df_feat["dz"] / df_feat["step_safe"]

df_feat["dx_prev"] = df_feat.groupby(group_cols)["dx"].shift(1)
df_feat["dy_prev"] = df_feat.groupby(group_cols)["dy"].shift(1)
df_feat["dz_prev"] = df_feat.groupby(group_cols)["dz"].shift(1)
df_feat["step_prev"] = df_feat.groupby(group_cols)["step"].shift(1)

df_feat["dx_prev_norm"] = df_feat["dx_prev"] / df_feat["step_prev"]
df_feat["dy_prev_norm"] = df_feat["dy_prev"] / df_feat["step_prev"]
df_feat["dz_prev_norm"] = df_feat["dz_prev"] / df_feat["step_prev"]

df_feat["curvature"] = (
    df_feat["dx_norm"] * df_feat["dx_prev_norm"] +
    df_feat["dy_norm"] * df_feat["dy_prev_norm"] +
    df_feat["dz_norm"] * df_feat["dz_prev_norm"]
)
df_feat["curvature"] = np.clip(df_feat["curvature"], -1.0, 1.0)
df_feat["turn_angle"] = np.arccos(df_feat["curvature"])

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)
df_feat.to_csv(FEATURE_OUT, index=False)

print("✅ Feature table rows:", len(df_feat))
print("✅ Feature columns:", feature_cols)
print("✅ Saved:", FEATURE_OUT)

display(df_feat[feature_cols].describe())


✅ Feature table rows: 6441646
✅ Feature columns: ['dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step']
✅ Saved: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv


,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
count,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06
mean,-1.177979e-03,-4.055961e-04,5.236998e-05,6.556832e-01,7.731258e-01,5.791190e+00
std,5.763636e-01,5.767268e-01,5.749628e-01,4.014457e-01,4.991009e-01,7.265077e-01
min,-9.999998e-01,-1.000000e+00,-9.999996e-01,-9.999973e-01,0.000000e+00,4.500008e+00
25%,-5.014891e-01,-5.015000e-01,-4.989068e-01,6.167521e-01,4.604168e-01,5.306649e+00
50%,-3.073370e-03,-1.161220e-03,-1.881477e-04,8.187188e-01,6.116201e-01,5.574335e+00
75%,4.980182e-01,4.998140e-01,4.992287e-01,8.958674e-01,9.061865e-01,6.020129e+00
max,9.999999e-01,1.000000e+00,9.999996e-01,1.000000e+00,3.139277e+00,7.999998e+00


In [5]:
# 5 — LOAD GEOMETRY FEATURE TABLE (DATASET — FINAL, MEMORY SAFE)

import pandas as pd
import gc
import os

# --- DATASET PATH (CONFIRMED STRUCTURE) ---
FEATURE_PATH = "/kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet"

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(f"❌ Feature table not found at:\n{FEATURE_PATH}")

print("📥 Loading geometry feature table...")
print("Path:", FEATURE_PATH)

# --- LOAD PARQUET (COLUMN-OPTIMIZED) ---
USE_COLS = [
    "target_id",
    "chain",
    "copy",
    "residue_index",
    "x", "y", "z",
    "dx", "dy", "dz",
    "dx_norm", "dy_norm", "dz_norm",
    "curvature",
    "turn_angle",
    "step"
]

df_feat = pd.read_parquet(FEATURE_PATH, columns=USE_COLS)

# --- TYPE OPTIMIZATION (CRITICAL FOR MEMORY) ---
float_cols = [
    "x","y","z",
    "dx","dy","dz",
    "dx_norm","dy_norm","dz_norm",
    "curvature","turn_angle","step"
]

for col in float_cols:
    df_feat[col] = df_feat[col].astype("float32")

df_feat["residue_index"] = df_feat["residue_index"].astype("int32")

# --- SORT FOR SEQUENTIAL OPERATIONS ---
df_feat = df_feat.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

# --- BASIC VALIDATION ---
print("\n✅ Feature table loaded")
print("Rows:", len(df_feat))
print("Targets:", df_feat["target_id"].nunique())
print("Chains:", df_feat.groupby(["target_id","chain","copy"]).ngroups)

print("\n📊 Sample:")
display(df_feat.head())

gc.collect()

📥 Loading geometry feature table...
Path: /kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet

✅ Feature table loaded
Rows: 7768829
Targets: 5744
Chains: 17746

📊 Sample:


,target_id,chain,copy,residue_index,x,y,z,dx,dy,dz,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
0,157D,A,1,2,3.385,-7.613,8.267,-1.458,-1.973,-4.998,-0.261871,-0.354370,-0.897689,0.521913,0.521913,5.567629
1,157D,A,1,3,2.158,-6.751,2.949,-1.227,0.862,-5.318,-0.222067,0.156008,-0.962470,0.392644,0.392644,5.525369
2,157D,A,1,4,2.669,-4.843,-1.773,0.511,1.908,-4.722,0.099834,0.372767,-0.922539,0.761646,0.761646,5.118483
3,157D,A,1,5,3.509,0.239,-4.045,0.840,5.082,-2.272,0.149207,0.902701,-0.403569,0.401361,0.401361,5.629770
4,157D,A,1,6,6.073,4.823,-5.035,2.564,4.584,-0.990,0.479716,0.857651,-0.185226,0.558137,0.558137,5.344834


0

In [6]:
# 6 — SAVE FEATURE TABLE (CRITICAL CHECKPOINT)

import os

FEATURE_PATH = "/kaggle/working/FEATURE_TABLE__GEOMETRY_FULL.parquet"

df_feat.to_parquet(FEATURE_PATH, index=False)

print("✅ Feature table saved")
print("Path:", FEATURE_PATH)
print("Rows:", len(df_feat))

✅ Feature table saved
Path: /kaggle/working/FEATURE_TABLE__GEOMETRY_FULL.parquet
Rows: 7768829


In [7]:
# 7 — MEMORY RESET BEFORE TRAINING

import gc

del df_labels
gc.collect()

print("🧹 Memory cleared — ready for training")

🧹 Memory cleared — ready for training


In [8]:
# 8 — BUILD TRAINING SET (DOUBLE-STEP FILTER — FINAL FIX)

import numpy as np

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

# --- TARGETS ---
df_feat["dx_target"] = df_feat.groupby(["target_id", "chain", "copy"])["dx"].shift(-1)
df_feat["dy_target"] = df_feat.groupby(["target_id", "chain", "copy"])["dy"].shift(-1)
df_feat["dz_target"] = df_feat.groupby(["target_id", "chain", "copy"])["dz"].shift(-1)

# --- COMPUTE NEXT STEP ---
df_feat["step_target"] = np.sqrt(
    df_feat["dx_target"]**2 +
    df_feat["dy_target"]**2 +
    df_feat["dz_target"]**2
)

# --- DROP NA ---
df_model = df_feat.dropna(
    subset=feature_cols + ["dx_target", "dy_target", "dz_target", "step_target"]
).copy()

print("Before filtering:", len(df_model))

# 🔥 FILTER CURRENT STEP
df_model = df_model[
    (df_model["step"] > 4.0) &
    (df_model["step"] < 8.0)
]

print("After current step filter:", len(df_model))

# 🔥 FILTER TARGET STEP (THIS IS THE FIX)
df_model = df_model[
    (df_model["step_target"] > 4.0) &
    (df_model["step_target"] < 8.0)
]

print("After target step filter:", len(df_model))

# 🔥 ANGLE SAFETY
df_model = df_model[
    (df_model["turn_angle"] >= 0.0) &
    (df_model["turn_angle"] <= np.pi)
]

print("After angle filter:", len(df_model))

# 🔥 SAMPLE
SAMPLE_SIZE = 1_500_000

if len(df_model) > SAMPLE_SIZE:
    df_model = df_model.sample(SAMPLE_SIZE, random_state=42)

print("Final training rows:", len(df_model))

# --- FINAL SANITY ---
print("\n📊 STEP CHECK:")
print(df_model["step"].describe())

print("\n📊 TARGET STEP CHECK:")
print(df_model["step_target"].describe())

print("\n📊 TARGET DELTA CHECK:")
print(df_model[["dx_target","dy_target","dz_target"]].describe())

Before filtering: 7237497
After current step filter: 6471404
After target step filter: 5919187
After angle filter: 5919187
Final training rows: 1500000

📊 STEP CHECK:
count    1.500000e+06
mean     5.743347e+00
std      6.997548e-01
min      4.000072e+00
25%      5.294896e+00
50%      5.553854e+00
75%      5.958226e+00
max      7.999993e+00
Name: step, dtype: float64

📊 TARGET STEP CHECK:
count    1.500000e+06
mean     5.748529e+00
std      7.086253e-01
min      4.000021e+00
25%      5.294417e+00
50%      5.551465e+00
75%      5.959690e+00
max      7.999987e+00
Name: step_target, dtype: float64

📊 TARGET DELTA CHECK:
          dx_target     dy_target     dz_target
count  1.500000e+06  1.500000e+06  1.500000e+06
mean  -1.290239e-02 -2.627963e-03  7.458999e-04
std    3.343432e+00  3.352167e+00  3.333618e+00
min   -7.987991e+00 -7.991005e+00 -7.987000e+00
25%   -2.849998e+00 -2.853027e+00 -2.820999e+00
50%   -2.999878e-02 -1.299667e-02  1.998901e-03
75%    2.822998e+00  2.839005e+00  2.82

In [9]:
# 9 — LOAD TRAINED MODELS (FINAL — TRUE PATH)

import joblib

MODEL_PATH = "/kaggle/input/datasets/pharaohtut/helix-rna-models-v1"

print("📦 Loading models from:", MODEL_PATH)

model_dx = joblib.load(f"{MODEL_PATH}/model_dx.pkl")
model_dy = joblib.load(f"{MODEL_PATH}/model_dy.pkl")
model_dz = joblib.load(f"{MODEL_PATH}/model_dz.pkl")

print("✅ Models loaded successfully")

📦 Loading models from: /kaggle/input/datasets/pharaohtut/helix-rna-models-v1
✅ Models loaded successfully


In [10]:
# 10 — BUILD EVALUATION SUBSET (CHAIN-LEVEL HOLDOUT, MEMORY-SAFE)

import pandas as pd
import numpy as np
import gc

group_cols = ["target_id", "chain", "copy"]

# Use the FULL feature table for evaluation candidate chains,
# but only evaluate on a limited number of chains to stay stable.
eval_keys = (
    df_feat[group_cols]
    .drop_duplicates()
    .sample(n=min(250, df_feat[group_cols].drop_duplicates().shape[0]), random_state=42)
    .reset_index(drop=True)
)

df_eval = df_feat.merge(eval_keys, on=group_cols, how="inner")

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_eval["dx_target"] = df_eval.groupby(group_cols)["dx"].shift(-1)
df_eval["dy_target"] = df_eval.groupby(group_cols)["dy"].shift(-1)
df_eval["dz_target"] = df_eval.groupby(group_cols)["dz"].shift(-1)

df_eval = df_eval.dropna(subset=feature_cols + ["dx_target", "dy_target", "dz_target"]).copy()
df_eval = df_eval.sort_values(group_cols + ["residue_index"]).reset_index(drop=True)

print("✅ Evaluation chains:", eval_keys.shape[0])
print("✅ Evaluation rows:", len(df_eval))

gc.collect()

✅ Evaluation chains: 250
✅ Evaluation rows: 109392


0

In [11]:
# 11 — STRUCTURAL VARIABILITY ENGINE (FINAL — ID SAFE + PARSE SAFE)

import hashlib
import numpy as np
import pandas as pd

sample = pd.read_csv(SAMPLE_SUB_PATH)
test_df = pd.read_csv(TEST_SEQS_PATH)

ENSEMBLE_SIZE = 5
TARGET_STEP = np.float32(5.82)
EPS = np.float32(1e-8)

print("🧬 Building structural-variability submission (final)...")

def normalize(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    if n < EPS:
        return np.array([1.0, 0.0, 0.0], dtype=np.float32)
    return v / n

def stable_seed(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)

def orthogonal_unit(v, rng):
    v = normalize(v)
    ref = np.array([0,0,1], dtype=np.float32)
    if abs(np.dot(v, ref)) > 0.9:
        ref = np.array([0,1,0], dtype=np.float32)
    u = normalize(np.cross(v, ref))
    if rng.random() < 0.5:
        u = -u
    return u

# --- PRECOMPUTE STRUCTURES ---
target_structures = {}

for _, row in test_df.iterrows():
    target_id = row["target_id"]
    seq = row["sequence"]
    L = len(seq)

    ensemble_coords = []

    for m in range(ENSEMBLE_SIZE):
        rng = np.random.default_rng(stable_seed(f"{target_id}_{m}"))

        coords = np.zeros((L,3), dtype=np.float32)
        direction = normalize(np.array([1.0, 0.2*m, 0.1*(m-2)], dtype=np.float32))
        phase = rng.uniform(0, 2*np.pi)

        for i in range(1, L):
            ortho1 = orthogonal_unit(direction, rng)
            ortho2 = normalize(np.cross(direction, ortho1))

            curve = (
                np.sin(phase)*ortho1 +
                0.3*np.cos(phase*0.7)*ortho2
            )

            if i % (10 + m*2) == 0:
                curve += 0.3 * ortho1

            if i % (15 + m*3) in (0,1):
                curve += 0.2 * (np.sin(phase)*ortho1 + np.cos(phase)*ortho2)

            direction = normalize(0.92*direction + 0.5*curve)
            coords[i] = coords[i-1] + direction * TARGET_STEP

            phase += 0.4 + 0.1*m

        ensemble_coords.append(coords)

    target_structures[target_id] = ensemble_coords

# --- BUILD SUBMISSION FROM SAMPLE ---
submission = sample.copy()

# 🔥 FIX dtype BEFORE assignment
coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]
submission[coord_cols] = submission[coord_cols].astype(np.float32)

for idx, row in submission.iterrows():
    full_id = row["ID"]

    # 🔥 CRITICAL FIX — RIGHT SPLIT
    target_id, resid = full_id.rsplit("_", 1)
    resid = int(resid) - 1

    for m in range(ENSEMBLE_SIZE):
        coords = target_structures[target_id][m][resid]

        submission.at[idx, f"x_{m+1}"] = coords[0]
        submission.at[idx, f"y_{m+1}"] = coords[1]
        submission.at[idx, f"z_{m+1}"] = coords[2]

print("✅ Submission built — TRUE ID alignment preserved")
display(submission.head())

🧬 Building structural-variability submission (final)...
✅ Submission built — TRUE ID alignment preserved


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,8ZNQ_2,C,2,5.781868,-0.660249,-0.080530,5.281640,-2.145057,-1.172778,5.683887,0.079049,-1.248833,2.495201,5.088414,1.324544,5.034246,2.338168,1.749782
2,8ZNQ_3,C,3,11.572744,-0.633108,0.500340,11.087997,-1.985963,-1.537882,10.639547,-2.493421,-2.890854,2.383526,10.584443,3.235982,9.166282,5.664666,4.144170
3,8ZNQ_4,G,4,17.130241,-2.345828,0.269652,16.713131,-0.504068,-1.352767,13.716625,-7.138028,-4.573594,3.482285,16.132906,4.606999,10.892150,10.425543,7.012589
4,8ZNQ_5,U,5,22.865856,-1.579124,0.891984,22.296368,1.137359,-1.427452,18.220562,-10.410608,-6.269815,5.456372,21.412062,6.058171,9.928561,15.183314,10.223121


In [12]:
# 12 — VALIDATE AND SAVE FINAL SUBMISSION (STRICT)

import numpy as np
import pandas as pd

OUT_PATH = "/kaggle/working/submission.csv"

sample = pd.read_csv(SAMPLE_SUB_PATH)

if list(submission.columns) != list(sample.columns):
    raise ValueError(
        "❌ Submission columns do not exactly match sample_submission.csv\n"
        f"Expected:\n{list(sample.columns)}\n\n"
        f"Got:\n{list(submission.columns)}"
    )

if len(submission) != len(sample):
    raise ValueError(
        f"❌ Submission row count mismatch: expected {len(sample)}, got {len(submission)}"
    )

if not submission["ID"].astype(str).equals(sample["ID"].astype(str)):
    raise ValueError("❌ Submission IDs do not exactly match sample_submission.csv")

coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]

if submission[coord_cols].isnull().any().any():
    bad = submission[coord_cols].isnull().sum()
    raise ValueError(f"❌ Null values found in coordinate columns:\n{bad[bad > 0]}")

vals = submission[coord_cols].to_numpy(dtype=np.float64)
if not np.isfinite(vals).all():
    raise ValueError("❌ Non-finite values found in coordinate columns")

submission["resname"] = submission["resname"].astype(str)
submission["resid"] = pd.to_numeric(submission["resid"], errors="raise").astype(np.int32)
for c in coord_cols:
    submission[c] = pd.to_numeric(submission[c], errors="raise").astype(np.float32)

submission.to_csv(OUT_PATH, index=False)

print("✅ submission.csv saved at:", OUT_PATH)
print("Shape:", submission.shape)
print("Coordinate dtypes:")
print(submission[coord_cols].dtypes.head())

✅ submission.csv saved at: /kaggle/working/submission.csv
Shape: (9762, 18)
Coordinate dtypes:
x_1    float32
y_1    float32
z_1    float32
x_2    float32
y_2    float32
dtype: object


In [13]:
# DIAG — CHECK WHAT EXISTS

print("df_submission in globals:", "df_submission" in globals())

for var in list(globals().keys()):
    if "sub" in var.lower():
        print("FOUND:", var)

df_submission in globals: False
FOUND: SAMPLE_SUB_PATH
FOUND: submission


In [14]:
# 12 — VALIDATE AND SAVE FINAL SUBMISSION (STRICT — SCHEMA LOCKED)

import numpy as np
import pandas as pd

OUT_PATH = "/kaggle/working/submission.csv"

print("🔍 Running strict submission validation...")

sample = pd.read_csv(SAMPLE_SUB_PATH)

# --- HARD SCHEMA LOCK ---
expected_cols = list(sample.columns)

missing_cols = [c for c in expected_cols if c not in submission.columns]
if missing_cols:
    raise ValueError(f"❌ Submission is missing required columns: {missing_cols}")

extra_cols = [c for c in submission.columns if c not in expected_cols]
if extra_cols:
    print(f"⚠️ Dropping extra columns not allowed by sample_submission: {extra_cols}")

# Keep only exact sample columns and exact order
submission = submission.loc[:, expected_cols].copy()

# --- BASIC STRUCTURE CHECKS ---
if list(submission.columns) != expected_cols:
    raise ValueError(
        "❌ Column mismatch after schema lock\n\n"
        f"Expected:\n{expected_cols}\n\n"
        f"Got:\n{list(submission.columns)}"
    )

if len(submission) != len(sample):
    raise ValueError(
        f"❌ Row count mismatch: expected {len(sample)}, got {len(submission)}"
    )

# --- ID VALIDATION ---
sub_ids = submission["ID"].astype(str).str.strip().reset_index(drop=True)
sample_ids = sample["ID"].astype(str).str.strip().reset_index(drop=True)

if not sub_ids.equals(sample_ids):
    mismatches = (sub_ids != sample_ids)
    mismatch_indices = np.where(mismatches)[0][:10]

    debug_rows = pd.DataFrame({
        "submission_ID": sub_ids.iloc[mismatch_indices].tolist(),
        "sample_ID": sample_ids.iloc[mismatch_indices].tolist(),
    })

    raise ValueError(
        "❌ Submission IDs do NOT match sample_submission.csv\n\n"
        f"First mismatches:\n{debug_rows}"
    )

print("✅ ID alignment PASSED")

# --- COORDINATE VALIDATION ---
coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]

if len(coord_cols) == 0:
    raise ValueError("❌ No coordinate columns found (x_*, y_*, z_*)")

# Force numeric before validation
for c in coord_cols:
    submission[c] = pd.to_numeric(submission[c], errors="raise")

if submission[coord_cols].isnull().any().any():
    bad = submission[coord_cols].isnull().sum()
    raise ValueError(f"❌ Null values detected:\n{bad[bad > 0]}")

vals = submission[coord_cols].to_numpy(dtype=np.float64)
if not np.isfinite(vals).all():
    raise ValueError("❌ Non-finite values found (NaN or Inf)")

print("✅ Coordinate validity PASSED")

# --- TYPE ENFORCEMENT ---
submission["resname"] = submission["resname"].astype(str)
submission["resid"] = pd.to_numeric(submission["resid"], errors="raise").astype(np.int32)
for c in coord_cols:
    submission[c] = submission[c].astype(np.float32)

print("✅ Dtype enforcement PASSED")

# --- FINAL SAVE ---
submission.to_csv(OUT_PATH, index=False)

print("\n🚀 SUBMISSION READY")
print("📁 Saved to:", OUT_PATH)
print("📐 Shape:", submission.shape)
print("📊 Coordinate dtype sample:")
print(submission[coord_cols].dtypes.head())

🔍 Running strict submission validation...
✅ ID alignment PASSED
✅ Coordinate validity PASSED
✅ Dtype enforcement PASSED

🚀 SUBMISSION READY
📁 Saved to: /kaggle/working/submission.csv
📐 Shape: (9762, 18)
📊 Coordinate dtype sample:
x_1    float32
y_1    float32
z_1    float32
x_2    float32
y_2    float32
dtype: object


In [15]:
# DIAG_13 — WINDOWED RECONSTRUCTION BENCHMARK (DRIFT-CONTROLLED)
import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZE = 10

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains with window size = {WINDOW_SIZE}")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < 4:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)
    true_steps = group["step"].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    coords_pred = np.zeros_like(coords_true, dtype=np.float32)
    coords_pred[0] = coords_true[0].copy()

    # Windowed reconstruction:
    # Reset anchor to true coordinate at each window start to measure local directional quality
    for start in range(0, len(group), WINDOW_SIZE):
        end = min(start + WINDOW_SIZE, len(group))

        coords_pred[start] = coords_true[start].copy()

        for i in range(start + 1, end):
            direction = np.array(
                [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                dtype=np.float32
            )

            norm = np.linalg.norm(direction)

            if norm < 1e-6 or not np.isfinite(norm):
                fallback = np.array(
                    [group.iloc[i - 1]["dx"], group.iloc[i - 1]["dy"], group.iloc[i - 1]["dz"]],
                    dtype=np.float32
                )
                fallback_norm = np.linalg.norm(fallback)

                if fallback_norm < 1e-6 or not np.isfinite(fallback_norm):
                    direction = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                    norm = 1.0
                else:
                    direction = fallback
                    norm = fallback_norm

            direction = direction / norm
            step_vec = direction * true_steps[i - 1]

            coords_pred[i] = coords_pred[i - 1] + step_vec

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "window_size": int(WINDOW_SIZE),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, true_steps
    gc.collect()

df_recon_windowed = pd.DataFrame(recon_rows)

print("\n🧬 WINDOWED RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon_windowed))
print("Mean error:", float(df_recon_windowed["mean_error"].mean()))
print("Median error:", float(df_recon_windowed["median_error"].median()))
print("P90:", float(np.percentile(df_recon_windowed["mean_error"], 90)))

display(df_recon_windowed.head(10))

🚀 Evaluating 249 chains with window size = 10
⏳ Evaluated 25/249
⏳ Evaluated 50/249
⏳ Evaluated 75/249
⏳ Evaluated 100/249
⏳ Evaluated 125/249
⏳ Evaluated 150/249
⏳ Evaluated 175/249
⏳ Evaluated 200/249
⏳ Evaluated 225/249

🧬 WINDOWED RECONSTRUCTION RESULTS
Chains evaluated: 241
Mean error: 8.346596471501584
Median error: 7.784458637237549
P90: 10.74055290222168


,target_id,chain,copy,residue_count,window_size,mean_error,median_error,p90_error
0,1A34,B,58,7,10,6.038911,7.610549,9.066518
1,1A34,C,3,7,10,5.999165,6.706511,9.236884
2,1A34,C,41,7,10,5.741069,6.602721,8.902356
3,1A34,C,43,7,10,5.489152,6.471849,8.360342
4,1A34,C,7,7,10,5.479325,6.209427,8.495967
5,1A3M,B,1,10,10,5.138705,4.327818,9.171608
6,1AQ3,R,12,14,10,9.492376,7.179642,17.857401
7,1AQ3,R,4,14,10,6.992448,6.371186,12.606200
8,1AQ4,R,120,12,10,11.124545,12.282446,20.730389
9,1AQ4,R,41,14,10,7.120039,7.036673,11.710606


In [16]:
# 13 — CORRECT KAGGLE SUBMISSION (MATCH SAMPLE FORMAT)

import numpy as np
import pandas as pd

print("📥 Loading sample submission...")

df_sample = pd.read_csv(SAMPLE_SUB_PATH)

print("Rows expected:", len(df_sample))

submission = df_sample.copy()

coords = np.zeros((len(submission), 3), dtype=np.float32)

print("🚀 Filling predictions...")

# --- SIMPLE BASELINE (valid, not competitive) ---
for i in range(len(submission)):

    if i == 0:
        coords[i] = np.array([0.0, 0.0, 0.0])
    else:
        coords[i] = coords[i-1] + np.array([5.7, 0.0, 0.0])

submission["x"] = coords[:, 0]
submission["y"] = coords[:, 1]
submission["z"] = coords[:, 2]

# --- SAVE ---
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("\n✅ Submission file created:", submission_path)
print("Shape:", submission.shape)

display(submission.head())

📥 Loading sample submission...
Rows expected: 9762
🚀 Filling predictions...

✅ Submission file created: /kaggle/working/submission.csv
Shape: (9762, 21)


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,...,z_3,x_4,y_4,z_4,x_5,y_5,z_5,x,y,z
0,8ZNQ_1,A,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.000000,0.0,0.0
1,8ZNQ_2,C,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,5.700000,0.0,0.0
2,8ZNQ_3,C,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,11.400000,0.0,0.0
3,8ZNQ_4,G,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,17.100000,0.0,0.0
4,8ZNQ_5,U,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,22.800001,0.0,0.0


In [17]:
# 14 — SLIDING WINDOW RECONSTRUCTION (OVERLAP + AVERAGE STITCHING)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZE = 4
STRIDE = 1

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains with sliding windows")
print(f"Window size = {WINDOW_SIZE}, stride = {STRIDE}")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < WINDOW_SIZE:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)
    true_steps = group["step"].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    n = len(group)

    # Accumulate multiple overlapping coordinate votes per residue
    coord_sum = np.zeros((n, 3), dtype=np.float64)
    coord_count = np.zeros(n, dtype=np.float64)

    window_starts = list(range(0, n - WINDOW_SIZE + 1, STRIDE))
    if window_starts[-1] != n - WINDOW_SIZE:
        window_starts.append(n - WINDOW_SIZE)

    for start in window_starts:
        end = start + WINDOW_SIZE

        # Anchor each window at its true start coordinate
        window_coords = [coords_true[start].astype(np.float32).copy()]

        for i in range(start + 1, end):
            direction = np.array(
                [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                dtype=np.float32
            )

            norm = np.linalg.norm(direction)

            if norm < 1e-6 or not np.isfinite(norm):
                fallback = np.array(
                    [group.iloc[i - 1]["dx"], group.iloc[i - 1]["dy"], group.iloc[i - 1]["dz"]],
                    dtype=np.float32
                )
                fallback_norm = np.linalg.norm(fallback)

                if fallback_norm < 1e-6 or not np.isfinite(fallback_norm):
                    direction = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                    norm = 1.0
                else:
                    direction = fallback
                    norm = fallback_norm

            direction = direction / norm
            step_vec = direction * true_steps[i - 1]

            window_coords.append(window_coords[-1] + step_vec)

        window_coords = np.asarray(window_coords, dtype=np.float32)

        # Stitch by averaging overlapping coordinate votes
        coord_sum[start:end] += window_coords.astype(np.float64)
        coord_count[start:end] += 1.0

    # Safety fallback for uncovered residues
    uncovered = coord_count == 0
    if np.any(uncovered):
        coord_sum[uncovered] = coords_true[uncovered]
        coord_count[uncovered] = 1.0

    coords_pred = (coord_sum / coord_count[:, None]).astype(np.float32)

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "window_size": int(WINDOW_SIZE),
        "stride": int(STRIDE),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, true_steps, coord_sum, coord_count
    gc.collect()

df_recon_sliding = pd.DataFrame(recon_rows)

print("\n🧬 SLIDING WINDOW RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon_sliding))
print("Mean error:", float(df_recon_sliding["mean_error"].mean()))
print("Median error:", float(df_recon_sliding["median_error"].median()))
print("P90:", float(np.percentile(df_recon_sliding["mean_error"], 90)))

display(df_recon_sliding.head(10))

SLIDING_OUT = "/kaggle/working/HELIX_SLIDING_WINDOW_RECON_V1.csv"
df_recon_sliding.to_csv(SLIDING_OUT, index=False)

print("\n💾 Saved:")
print(SLIDING_OUT)

🚀 Evaluating 249 chains with sliding windows
Window size = 4, stride = 1
⏳ Evaluated 25/249
⏳ Evaluated 50/249
⏳ Evaluated 75/249
⏳ Evaluated 100/249
⏳ Evaluated 125/249
⏳ Evaluated 150/249
⏳ Evaluated 175/249
⏳ Evaluated 200/249
⏳ Evaluated 225/249

🧬 SLIDING WINDOW RECONSTRUCTION RESULTS
Chains evaluated: 241
Mean error: 4.412763618334695
Median error: 3.881382465362549
P90: 5.20732307434082


,target_id,chain,copy,residue_count,window_size,stride,mean_error,median_error,p90_error
0,1A34,B,58,7,4,1,3.987112,3.958905,7.496825
1,1A34,C,3,7,4,1,4.176062,3.765867,7.508367
2,1A34,C,41,7,4,1,3.926873,3.855507,6.893155
3,1A34,C,43,7,4,1,3.780617,3.753931,6.571071
4,1A34,C,7,7,4,1,3.904819,3.588330,7.126676
5,1A3M,B,1,10,4,1,3.492925,2.835690,6.520891
6,1AQ3,R,12,14,4,1,4.412869,4.389615,6.287423
7,1AQ3,R,4,14,4,1,4.161301,4.393273,6.274519
8,1AQ4,R,120,12,4,1,5.169311,5.881486,8.233427
9,1AQ4,R,41,14,4,1,3.750844,4.160289,5.633806



💾 Saved:
/kaggle/working/HELIX_SLIDING_WINDOW_RECON_V1.csv


In [18]:
# 15 — FULL GENERATIVE RECONSTRUCTION (NO TRUE STEP, KAGGLE-DEPLOYABLE)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZE = 4
STRIDE = 1

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains in fully generative mode")
print(f"Window size = {WINDOW_SIZE}, stride = {STRIDE}")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < WINDOW_SIZE:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    n = len(group)

    coord_sum = np.zeros((n, 3), dtype=np.float64)
    coord_count = np.zeros(n, dtype=np.float64)

    window_starts = list(range(0, n - WINDOW_SIZE + 1, STRIDE))
    if window_starts[-1] != n - WINDOW_SIZE:
        window_starts.append(n - WINDOW_SIZE)

    for start in window_starts:
        end = start + WINDOW_SIZE

        # Anchor each window at its true start only for evaluation alignment.
        # For test-time deployment, this anchor must come from your chosen initialization strategy.
        window_coords = [coords_true[start].astype(np.float32).copy()]

        for i in range(start + 1, end):
            step_vec = np.array(
                [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                dtype=np.float32
            )

            if not np.all(np.isfinite(step_vec)):
                step_vec = np.zeros(3, dtype=np.float32)

            coords_next = window_coords[-1] + step_vec
            window_coords.append(coords_next)

        window_coords = np.asarray(window_coords, dtype=np.float32)

        coord_sum[start:end] += window_coords.astype(np.float64)
        coord_count[start:end] += 1.0

    uncovered = coord_count == 0
    if np.any(uncovered):
        coord_sum[uncovered] = coords_true[uncovered]
        coord_count[uncovered] = 1.0

    coords_pred = (coord_sum / coord_count[:, None]).astype(np.float32)

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "window_size": int(WINDOW_SIZE),
        "stride": int(STRIDE),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, coord_sum, coord_count
    gc.collect()

df_recon_generative = pd.DataFrame(recon_rows)

print("\n🧬 FULL GENERATIVE RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon_generative))
print("Mean error:", float(df_recon_generative["mean_error"].mean()))
print("Median error:", float(df_recon_generative["median_error"].median()))
print("P90:", float(np.percentile(df_recon_generative["mean_error"], 90)))

display(df_recon_generative.head(10))

GENERATIVE_OUT = "/kaggle/working/HELIX_GENERATIVE_RECON_V1.csv"
df_recon_generative.to_csv(GENERATIVE_OUT, index=False)

print("\n💾 Saved:")
print(GENERATIVE_OUT)

🚀 Evaluating 249 chains in fully generative mode
Window size = 4, stride = 1
⏳ Evaluated 25/249
⏳ Evaluated 50/249
⏳ Evaluated 75/249
⏳ Evaluated 100/249
⏳ Evaluated 125/249
⏳ Evaluated 150/249
⏳ Evaluated 175/249
⏳ Evaluated 200/249
⏳ Evaluated 225/249

🧬 FULL GENERATIVE RECONSTRUCTION RESULTS
Chains evaluated: 241
Mean error: 4.071233821607724
Median error: 3.749837875366211
P90: 4.614885330200195


,target_id,chain,copy,residue_count,window_size,stride,mean_error,median_error,p90_error
0,1A34,B,58,7,4,1,3.843089,3.802827,7.159502
1,1A34,C,3,7,4,1,4.033226,3.797294,7.234222
2,1A34,C,41,7,4,1,3.740613,3.752786,6.578182
3,1A34,C,43,7,4,1,3.642279,3.620432,6.426051
4,1A34,C,7,7,4,1,3.741297,3.537577,6.806478
5,1A3M,B,1,10,4,1,3.249642,2.748078,5.675675
6,1AQ3,R,12,14,4,1,3.947038,4.316172,5.664226
7,1AQ3,R,4,14,4,1,3.913166,4.428850,5.378669
8,1AQ4,R,120,12,4,1,4.419611,4.696283,6.780682
9,1AQ4,R,41,14,4,1,3.761387,3.783031,5.255328



💾 Saved:
/kaggle/working/HELIX_GENERATIVE_RECON_V1.csv


In [19]:
# 16 — FULL KAGGLE SUBMISSION PIPELINE (SEQUENCE → STRUCTURE)

import numpy as np
import pandas as pd
import gc

# =========================
# LOAD TEST DATA
# =========================
df_test = pd.read_csv(TEST_SEQS_PATH)

if "target_id" not in df_test.columns:
    df_test["target_id"] = df_test["ID"].astype(str)

print("✅ Test rows:", len(df_test))

# =========================
# PARAMETERS
# =========================
WINDOW_SIZE = 4
STRIDE = 1

# =========================
# HELPER: BUILD FEATURES FROM PREDICTED TRAJECTORY
# =========================
def build_features_from_coords(coords):
    coords = np.asarray(coords, dtype=np.float32)
    n = len(coords)

    dx = np.zeros(n, dtype=np.float32)
    dy = np.zeros(n, dtype=np.float32)
    dz = np.zeros(n, dtype=np.float32)
    step = np.zeros(n, dtype=np.float32)

    dx[1:] = coords[1:,0] - coords[:-1,0]
    dy[1:] = coords[1:,1] - coords[:-1,1]
    dz[1:] = coords[1:,2] - coords[:-1,2]

    step[1:] = np.sqrt(dx[1:]**2 + dy[1:]**2 + dz[1:]**2)

    # normalize direction
    dx_norm = np.zeros(n, dtype=np.float32)
    dy_norm = np.zeros(n, dtype=np.float32)
    dz_norm = np.zeros(n, dtype=np.float32)

    valid = step > 1e-6
    dx_norm[valid] = dx[valid] / step[valid]
    dy_norm[valid] = dy[valid] / step[valid]
    dz_norm[valid] = dz[valid] / step[valid]

    # curvature (cosine of angle)
    curvature = np.zeros(n, dtype=np.float32)
    turn_angle = np.zeros(n, dtype=np.float32)

    for i in range(2, n):
        v1 = np.array([dx[i-1], dy[i-1], dz[i-1]])
        v2 = np.array([dx[i], dy[i], dz[i]])

        n1 = np.linalg.norm(v1)
        n2 = np.linalg.norm(v2)

        if n1 > 1e-6 and n2 > 1e-6:
            cos_angle = np.dot(v1, v2) / (n1 * n2)
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            curvature[i] = cos_angle
            turn_angle[i] = np.arccos(cos_angle)

    df_feat = pd.DataFrame({
        "dx_norm": dx_norm,
        "dy_norm": dy_norm,
        "dz_norm": dz_norm,
        "curvature": curvature,
        "turn_angle": turn_angle,
        "step": step
    })

    return df_feat.fillna(0.0)


# =========================
# MAIN GENERATION LOOP
# =========================
submission_rows = []

print("🚀 Generating structures...")

for idx, row in df_test.iterrows():

    target_id = row["target_id"]
    seq = row["sequence"]

    L = len(seq)

    # initial straight-line bootstrap
    coords = np.zeros((L, 3), dtype=np.float32)
    for i in range(1, L):
        coords[i] = coords[i-1] + np.array([5.7, 0, 0], dtype=np.float32)

    # iterative refinement (2 passes)
    for _ in range(2):

        df_feat = build_features_from_coords(coords)
        X = df_feat[["dx_norm","dy_norm","dz_norm","curvature","turn_angle","step"]].values.astype(np.float32)

        pred_dx = model_dx.predict(X).astype(np.float32)
        pred_dy = model_dy.predict(X).astype(np.float32)
        pred_dz = model_dz.predict(X).astype(np.float32)

        # sliding window stitching
        coord_sum = np.zeros_like(coords, dtype=np.float64)
        coord_count = np.zeros(L, dtype=np.float64)

        starts = list(range(0, L - WINDOW_SIZE + 1, STRIDE))
        if starts[-1] != L - WINDOW_SIZE:
            starts.append(L - WINDOW_SIZE)

        for start in starts:
            end = start + WINDOW_SIZE

            window_coords = [coords[start].copy()]

            for i in range(start+1, end):
                step_vec = np.array([
                    pred_dx[i-1],
                    pred_dy[i-1],
                    pred_dz[i-1]
                ], dtype=np.float32)

                if not np.all(np.isfinite(step_vec)):
                    step_vec = np.zeros(3, dtype=np.float32)

                window_coords.append(window_coords[-1] + step_vec)

            window_coords = np.asarray(window_coords)

            coord_sum[start:end] += window_coords
            coord_count[start:end] += 1

        coords = (coord_sum / coord_count[:, None]).astype(np.float32)

    # =========================
    # STORE OUTPUT
    # =========================
    for i in range(L):
        submission_rows.append({
            "ID": f"{target_id}_{i+1}",
            "x": coords[i,0],
            "y": coords[i,1],
            "z": coords[i,2]
        })

    if idx % 10 == 0:
        print(f"⏳ Processed {idx}/{len(df_test)}")

    gc.collect()

# =========================
# SAVE SUBMISSION
# =========================
df_submission = pd.DataFrame(submission_rows)

SUB_PATH = "/kaggle/working/submission.csv"
df_submission.to_csv(SUB_PATH, index=False)

print("\n✅ SUBMISSION FILE CREATED")
print(SUB_PATH)
print("Rows:", len(df_submission))

display(df_submission.head())

✅ Test rows: 28
🚀 Generating structures...
⏳ Processed 0/28
⏳ Processed 10/28
⏳ Processed 20/28

✅ SUBMISSION FILE CREATED
/kaggle/working/submission.csv
Rows: 9762


,ID,x,y,z
0,8ZNQ_1,0.000000,0.000000,0.000000
1,8ZNQ_2,1.461895,-0.012511,-0.194237
2,8ZNQ_3,8.309007,0.063388,-0.654129
3,8ZNQ_4,13.083393,-0.286436,-0.561813
4,8ZNQ_5,18.927790,-0.414680,-0.493750


In [20]:
# 16b — SAFE SUBMISSION BUILD (STRICT SAMPLE ALIGNMENT)

import pandas as pd
import numpy as np

sample = pd.read_csv(SAMPLE_SUB_PATH)

print("✅ Sample rows:", len(sample))

# Build lookup from your generated coordinates
coord_lookup = {
    row["ID"]: (row["x"], row["y"], row["z"])
    for _, row in df_submission.iterrows()
}

final_rows = []
missing = 0

for _, row in sample.iterrows():
    ID = row["ID"]

    if ID in coord_lookup:
        x, y, z = coord_lookup[ID]
    else:
        # fallback (must exist)
        x, y, z = 0.0, 0.0, 0.0
        missing += 1

    final_rows.append({
        "ID": ID,
        "x": float(x),
        "y": float(y),
        "z": float(z)
    })

df_submission_fixed = pd.DataFrame(final_rows)

print("⚠️ Missing IDs filled:", missing)
print("Final rows:", len(df_submission_fixed))

# HARD VALIDATION (DO NOT SKIP)
assert len(df_submission_fixed) == len(sample), "Row count mismatch"
assert list(df_submission_fixed.columns) == ["ID", "x", "y", "z"], "Column mismatch"
assert df_submission_fixed["ID"].equals(sample["ID"]), "ID mismatch"

SUB_PATH = "/kaggle/working/submission.csv"
df_submission_fixed.to_csv(SUB_PATH, index=False)

print("\n✅ FINAL SUBMISSION READY")
print(SUB_PATH)

display(df_submission_fixed.head())

✅ Sample rows: 9762
⚠️ Missing IDs filled: 0
Final rows: 9762

✅ FINAL SUBMISSION READY
/kaggle/working/submission.csv


,ID,x,y,z
0,8ZNQ_1,0.000000,0.000000,0.000000
1,8ZNQ_2,1.461895,-0.012511,-0.194237
2,8ZNQ_3,8.309007,0.063388,-0.654129
3,8ZNQ_4,13.083393,-0.286436,-0.561813
4,8ZNQ_5,18.927790,-0.414680,-0.493750


In [21]:
sample = pd.read_csv(SAMPLE_SUB_PATH)
print("Sample rows:", len(sample))
print("Your rows:", len(df_submission))

Sample rows: 9762
Your rows: 9762


In [22]:
print(df_submission.isna().sum())
print(np.isfinite(df_submission[["x","y","z"]]).all())


ID    0
x     0
y     0
z     0
dtype: int64
x    True
y    True
z    True
dtype: bool


In [23]:
# DIAG_99 — FULL NOTEBOOK OUTPUT AUDIT (FIXED + ROBUST)

import sys
import numpy as np
import pandas as pd

print("\n" + "="*80)
print("🔍 HELIX FULL OUTPUT AUDIT (FINAL)")
print("="*80)

# ===============================
# 1. GLOBAL VARIABLES
# ===============================
print("\n📦 GLOBAL VARIABLES:")
global_vars = {k: v for k, v in globals().items() if not k.startswith("_")}

for k, v in global_vars.items():
    try:
        if isinstance(v, pd.DataFrame):
            print(f"🧾 {k}: DataFrame {v.shape}")
        elif isinstance(v, np.ndarray):
            print(f"🔢 {k}: ndarray {v.shape}")
        else:
            print(f"📌 {k}: {type(v)}")
    except:
        print(f"⚠️ {k}: <uninspectable>")

# ===============================
# 2. DATAFRAME DETAILS
# ===============================
print("\n📊 DATAFRAME DETAILS:")

for k, v in global_vars.items():
    if isinstance(v, pd.DataFrame):
        print("\n" + "-"*60)
        print(f"🧾 {k}")
        print("-"*60)
        print("Shape:", v.shape)
        print("\nColumns:", list(v.columns))
        print("\nDtypes:\n", v.dtypes)
        print("\nHead:")
        display(v.head(3))

# ===============================
# 3. NUMPY ARRAYS
# ===============================
print("\n🔢 NUMPY ARRAYS:")

for k, v in global_vars.items():
    if isinstance(v, np.ndarray):
        print(f"{k}: shape={v.shape}, dtype={v.dtype}")

# ===============================
# 4. MODEL CHECK
# ===============================
print("\n🤖 MODEL CHECK:")

for name in ["model_dx", "model_dy", "model_dz"]:
    if name in globals():
        print(f"✅ {name} loaded")
    else:
        print(f"❌ {name} MISSING")

# ===============================
# 5. SUBMISSION VALIDATION
# ===============================
print("\n📄 SUBMISSION CHECK:")

if "df_submission" in globals():
    sub = df_submission.copy()
    print("Shape:", sub.shape)
    
    required_cols = [
        "ID","resname","resid",
        "x_1","y_1","z_1",
        "x_2","y_2","z_2",
        "x_3","y_3","z_3",
        "x_4","y_4","z_4",
        "x_5","y_5","z_5"
    ]
    
    missing = [c for c in required_cols if c not in sub.columns]
    
    if missing:
        print("❌ Missing columns:", missing)
    else:
        print("✅ All required columns present")

    print("\nDtypes:")
    print(sub.dtypes.head())

    print("\nSample:")
    display(sub.head(5))

    # --- NaN check ---
    nan_count = sub.isna().sum().sum()
    print("\nNaN count:", nan_count)

else:
    print("❌ df_submission NOT FOUND")

# ===============================
# 6. COORDINATE STATS (FIXED)
# ===============================
print("\n📐 COORDINATE STATS:")

if "df_submission" in globals():
    sub = df_submission.copy()

    coord_cols = [c for c in sub.columns if c.startswith(("x_", "y_", "z_"))]

    if len(coord_cols) == 0:
        print("❌ No coordinate columns found")
        print("Available columns:")
        print(list(sub.columns))
    else:
        coords = sub[coord_cols]

        print(f"✅ Found {len(coord_cols)} coordinate columns")

        try:
            stats = coords.describe()
            print(stats.loc[["mean", "std", "min", "max"]])
        except Exception as e:
            print("⚠️ Could not compute describe:", e)

# ===============================
# 7. BOND LENGTH CHECK
# ===============================
print("\n🧬 BOND LENGTH CHECK:")

if "df_submission" in globals():
    sub = df_submission.copy()
    sub["target"] = sub["ID"].str.split("_").str[0]

    medians = []
    maxes = []

    for t, df in sub.groupby("target"):
        if all(c in df.columns for c in ["x_1","y_1","z_1"]):
            X = df[["x_1","y_1","z_1"]].values

            if len(X) > 1:
                d = np.linalg.norm(np.diff(X, axis=0), axis=1)
                medians.append(np.median(d))
                maxes.append(np.max(d))

    if medians:
        print("Median bond length:", np.median(medians))
        print("Worst bond length:", np.max(maxes))
    else:
        print("⚠️ Not enough valid coordinate data")

# ===============================
# 8. MEMORY USAGE
# ===============================
print("\n💾 MEMORY USAGE:")

total_mem = 0

for k, v in global_vars.items():
    try:
        if isinstance(v, pd.DataFrame):
            mem = v.memory_usage(deep=True).sum()
        elif isinstance(v, np.ndarray):
            mem = v.nbytes
        else:
            continue
        
        total_mem += mem
        print(f"{k}: {mem/1e6:.2f} MB")
    except:
        pass

print(f"\nTotal tracked memory: {total_mem/1e6:.2f} MB")

# ===============================
# 9. FINAL STATUS
# ===============================
print("\n" + "="*80)
print("✅ AUDIT COMPLETE — NO CRASHES")
print("="*80)


🔍 HELIX FULL OUTPUT AUDIT (FINAL)

📦 GLOBAL VARIABLES:
📌 In: <class 'list'>
📌 Out: <class 'dict'>
📌 get_ipython: <class 'method'>
📌 exit: <class 'IPython.core.autocall.ZMQExitAutocall'>
📌 quit: <class 'IPython.core.autocall.ZMQExitAutocall'>
📌 os: <class 'module'>
📌 gc: <class 'module'>
📌 np: <class 'module'>
📌 pd: <class 'module'>
📌 BASE_CANDIDATES: <class 'list'>
📌 COMP_ROOT: <class 'str'>
📌 p: <class 'str'>
📌 TRAIN_LABELS_PATH: <class 'str'>
📌 VALIDATION_LABELS_PATH: <class 'str'>
📌 TRAIN_SEQS_PATH: <class 'str'>
📌 VALIDATION_SEQS_PATH: <class 'str'>
📌 TEST_SEQS_PATH: <class 'str'>
📌 SAMPLE_SUB_PATH: <class 'str'>
📌 REQUIRED_FILES: <class 'list'>
📌 missing_files: <class 'list'>
📌 WORK_DIR: <class 'str'>
📌 FEATURE_OUT: <class 'str'>
📌 BENCH_OUT: <class 'str'>
📌 f: <class 'str'>
📌 USE_COLS: <class 'list'>
📌 DTYPES: <class 'dict'>
📌 group_cols: <class 'list'>
🧾 df_geo: DataFrame (6996848, 14)
🔢 steps: ndarray (6475882,)
🧾 df_feat: DataFrame (1460, 6)
📌 feature_cols: <class 'list'>
📌 F

,resname,chain,copy,target_id,residue_index,x,y,z,resid_diff,is_seq,dx,dy,dz,step
0,C,A,1,157D,1,4.843,-5.640,13.265,NaN,False,NaN,NaN,NaN,NaN
1,G,A,1,157D,2,3.385,-7.613,8.267,1.0,True,-1.458,-1.973,-4.998,5.567629
2,C,A,1,157D,3,2.158,-6.751,2.949,1.0,True,-1.227,0.862,-5.318,5.525369



------------------------------------------------------------
🧾 df_feat
------------------------------------------------------------
Shape: (1460, 6)

Columns: ['dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step']

Dtypes:
 dx_norm       float32
dy_norm       float32
dz_norm       float32
curvature     float32
turn_angle    float32
step          float32
dtype: object

Head:


,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
1,0.998983,-0.002899,-0.045001,0.00000,0.000000,2.877524
2,0.999502,0.010884,0.029630,0.99712,0.075912,6.700646



------------------------------------------------------------
🧾 df_model
------------------------------------------------------------
Shape: (1500000, 20)

Columns: ['target_id', 'chain', 'copy', 'residue_index', 'x', 'y', 'z', 'dx', 'dy', 'dz', 'dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step', 'dx_target', 'dy_target', 'dz_target', 'step_target']

Dtypes:
 target_id         object
chain             object
copy              object
residue_index      int32
x                float32
y                float32
z                float32
dx               float32
dy               float32
dz               float32
dx_norm          float32
dy_norm          float32
dz_norm          float32
curvature        float32
turn_angle       float32
step             float32
dx_target        float32
dy_target        float32
dz_target        float32
step_target      float32
dtype: object

Head:


...


------------------------------------------------------------
🧾 df_submission_fixed
------------------------------------------------------------
Shape: (9762, 4)

Columns: ['ID', 'x', 'y', 'z']

Dtypes:
 ID     object
x     float64
y     float64
z     float64
dtype: object

Head:


,ID,x,y,z
0,8ZNQ_1,0.000000,0.000000,0.000000
1,8ZNQ_2,1.461895,-0.012511,-0.194237
2,8ZNQ_3,8.309007,0.063388,-0.654129



🔢 NUMPY ARRAYS:
steps: shape=(6475882,), dtype=float32
coords: shape=(1460, 3), dtype=float32
direction: shape=(3,), dtype=float32
ortho1: shape=(3,), dtype=float32
ortho2: shape=(3,), dtype=float32
curve: shape=(3,), dtype=float64
vals: shape=(9762, 15), dtype=float64
step_vec: shape=(3,), dtype=float32
window_coords: shape=(4, 3), dtype=float32
uncovered: shape=(152,), dtype=bool
coords_next: shape=(3,), dtype=float32
X: shape=(1460, 6), dtype=float32
pred_dx: shape=(1460,), dtype=float32
pred_dy: shape=(1460,), dtype=float32
pred_dz: shape=(1460,), dtype=float32
coord_sum: shape=(1460, 3), dtype=float64
coord_count: shape=(1460,), dtype=float64

🤖 MODEL CHECK:
✅ model_dx loaded
✅ model_dy loaded
✅ model_dz loaded

📄 SUBMISSION CHECK:
Shape: (9762, 4)
❌ Missing columns: ['resname', 'resid', 'x_1', 'y_1', 'z_1', 'x_2', 'y_2', 'z_2', 'x_3', 'y_3', 'z_3', 'x_4', 'y_4', 'z_4', 'x_5', 'y_5', 'z_5']

Dtypes:
ID     object
x     float32
y     float32
z     float32
dtype: object

Sample:


,ID,x,y,z
0,8ZNQ_1,0.000000,0.000000,0.000000
1,8ZNQ_2,1.461895,-0.012511,-0.194237
2,8ZNQ_3,8.309007,0.063388,-0.654129
3,8ZNQ_4,13.083393,-0.286436,-0.561813
4,8ZNQ_5,18.927790,-0.414680,-0.493750



NaN count: 0

📐 COORDINATE STATS:
❌ No coordinate columns found
Available columns:
['ID', 'x', 'y', 'z']

🧬 BOND LENGTH CHECK:
⚠️ Not enough valid coordinate data

💾 MEMORY USAGE:
df_geo: 1423.35 MB
steps: 25.90 MB
df_feat: 0.04 MB
df_model: 344.13 MB
eval_keys: 0.04 MB
df_eval: 24.23 MB
sample: 2.30 MB
test_df: 0.03 MB
coords: 0.02 MB
direction: 0.00 MB
ortho1: 0.00 MB
ortho2: 0.00 MB
curve: 0.00 MB
submission: 2.41 MB
vals: 1.17 MB
group: 0.03 MB
step_vec: 0.00 MB
df_recon_windowed: 0.05 MB
df_sample: 2.30 MB
window_coords: 0.00 MB
uncovered: 0.00 MB
df_recon_sliding: 0.05 MB
coords_next: 0.00 MB
df_recon_generative: 0.05 MB
df_test: 0.03 MB
X: 0.04 MB
pred_dx: 0.01 MB
pred_dy: 0.01 MB
pred_dz: 0.01 MB
coord_sum: 0.04 MB
coord_count: 0.01 MB
df_submission: 0.68 MB
df_submission_fixed: 0.79 MB

Total tracked memory: 1827.69 MB

✅ AUDIT COMPLETE — NO CRASHES
